# Goal-Based Agent: Warehouse Navigation (BFS)

In [1]:
from collections import deque

WAREHOUSE = """
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
""".strip().splitlines()

In [2]:
class WarehouseAgent:
    """
    Goal-based agent using Breadth-First Search (BFS).

    BFS is appropriate because all moves cost 1 (uniform step cost), the grid is
    small and finite, and BFS is complete and returns a shortest path.
    """
    MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

    def __init__(self, layout):
        self.grid = [list(row) for row in layout]
        self.rows, self.cols = len(self.grid), len(self.grid[0])
        self.start = self._find("S")
        self.goal = self._find("G")

    def _find(self, symbol):
        for r, row in enumerate(self.grid):
            for c, ch in enumerate(row):
                if ch == symbol:
                    return (r, c)
        raise ValueError(f"Symbol {symbol!r} not found in map")

    def goal_test(self, state):
        return state == self.goal

    def actions(self, state):
        """Legal (action, next_state) pairs: inside the grid and not an obstacle."""
        r, c = state
        for name, (dr, dc) in self.MOVES.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols and self.grid[nr][nc] != "#":
                yield name, (nr, nc)

    def search(self):
        """Return (list_of_actions, list_of_states) or (None, None) if no path exists."""
        frontier = deque([self.start])
        parent = {self.start: (None, None)}  # state -> (previous_state, action); doubles as visited set
        expanded = 0
        while frontier:
            state = frontier.popleft()
            expanded += 1
            if self.goal_test(state):
                self.expanded = expanded
                return self._reconstruct(parent, state)
            for action, nxt in self.actions(state):
                if nxt not in parent:
                    parent[nxt] = (state, action)
                    frontier.append(nxt)
        self.expanded = expanded
        return None, None

    @staticmethod
    def _reconstruct(parent, state):
        actions, states = [], [state]
        while parent[state][0] is not None:
            prev, act = parent[state]
            actions.append(act); states.append(prev); state = prev
        return actions[::-1], states[::-1]

    def render(self, path_states=None):
        canvas = [row[:] for row in self.grid]
        for (r, c) in (path_states or [])[1:-1]:
            canvas[r][c] = "*"
        return "\n".join("".join(row) for row in canvas)

In [3]:
agent = WarehouseAgent(WAREHOUSE)
actions, states = agent.search()

print("Start:", agent.start, "| Goal:", agent.goal)
if actions is None:
    print("No collision-free path exists from S to G.")
else:
    print(f"Path found: {len(actions)} moves, {agent.expanded} states expanded\n")
    print("Actions:", " -> ".join(actions))
    print("States :", states, "\n")
    print(agent.render(states))

Start: (1, 1) | Goal: (1, 19)
Path found: 20 moves, 59 states expanded

Actions: Right -> Right -> Right -> Down -> Right -> Right -> Right -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right
States : [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)] 

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


## Tests

In [4]:
# 1. Path validity: starts at S, ends at G, unit steps, no obstacles
assert states[0] == agent.start and states[-1] == agent.goal
for (r1, c1), (r2, c2) in zip(states, states[1:]):
    assert abs(r1 - r2) + abs(c1 - c2) == 1
assert all(agent.grid[r][c] != "#" for r, c in states)

# 2. Actions replayed from S reach G
pos = agent.start
for a in actions:
    dr, dc = agent.MOVES[a]; pos = (pos[0] + dr, pos[1] + dc)
assert pos == agent.goal

# 3. Optimality: cross-check with an independent DP (Dijkstra with unit costs)
import heapq
dist, pq = {agent.start: 0}, [(0, agent.start)]
while pq:
    d, s = heapq.heappop(pq)
    if d > dist[s]: continue
    for _, n in agent.actions(s):
        if d + 1 < dist.get(n, 1e9):
            dist[n] = d + 1; heapq.heappush(pq, (d + 1, n))
assert len(actions) == dist[agent.goal]

# 4. No-path case: wall off the goal
blocked = [row for row in WAREHOUSE]
blocked[1] = blocked[1][:18] + "#G#"   # wall left of G
blocked[2] = blocked[2][:18] + "###"   # wall below G
no_path_agent = WarehouseAgent(blocked)
a, s = no_path_agent.search()
assert a is None
print("No-path case ->", "No collision-free path exists from S to G.")

# 5. Start == goal edge case
trivial = WarehouseAgent(["#####", "#S.G#", "#####"]); trivial.goal = trivial.start
assert trivial.search()[0] == []

print("All tests passed.")

No-path case -> No collision-free path exists from S to G.
All tests passed.
